# 1. Introduction

LLM models: there are two kinds of llm models from OpenAI, `completion-based models (e.g., gpt-3.5-turbo-instruct)` and `chat-based models (e.g., gpt-4-turbo)` Not every method or implementation works with each kind because honestly based on my experience, langchain is extremely difficult to use and maintain, there are still constant updates.

For ReAct prompting using Langchain and OpenAI, I identified three options right now. They are have different pros and cons, but generally, i prefer the second one.


* using DocstoreExplorer:
    * a lot of the tutorials still use this
    * **deprecated** but still can work
    * only works with completion-based models
    * `from langchain.agents.react.base import DocstoreExplorer`
    * `from langchain.agents import initialize_agent, Tool`
    * https://www.comet.com/site/blog/using-the-react-framework-in-langchain/

* using create_react_agent()
    * newest method of creating ReAct agent
    * `from langchain.agents import create_react_agent`
    * only works with completion-based models
    * https://medium.com/@BuzonXXXX/creating-a-react-agent-with-langchain-and-openai-d7e2c25d8d15

* using langGraph
    * `from langgraph.prebuilt import create_react_agent`
    * **works with chat-based models**
    * https://langchain-ai.github.io/langgraph/reference/prebuilt/?h=react


<font color='red'>Different method even use different 'tool' from different libraires of langchain.</font>

In [2]:
!pip -q install python-dotenv langchain langchain_openai langchain_community pypdf sentence-transformers PyPDF2 pycryptodome tiktoken faiss-gpu langgraph


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.2/292.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.0/383.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141

# 2. Read PDF and Build RAG Tool

In [7]:
%%writefile .env
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"

Writing .env


In [1]:
# test openai api key loaded
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import os
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
# from langchain.embeddings.openai import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain import FAISS

In [5]:
def process_text(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=400,
        chunk_overlap=100,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    embeddings = OpenAIEmbeddings()
    knowledgeBase = FAISS.from_texts(chunks, embeddings)
    return knowledgeBase

pdf_reader = PdfReader('/content/naf23.pdf')
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

knowledgeBase = process_text(text)

def faiss_search(query):
    return knowledgeBase.similarity_search(query, k=2)

In [ ]:
tools = [
    Tool(
        name="Search",
        func=faiss_search,
        description="Useful for searching through the document store based on keywords."
    ),
    Tool(name="Lookup",
        func=faiss_search,
        description="Useful for looking up specific information or facts."
    ),
]

# 3. Using Docstore

In [7]:
from langchain import OpenAI
from langchain.agents import initialize_agent, Tool
from langchain.agents import AgentType
from langchain.agents.react.base import DocstoreExplorer

search_tool = Tool(name="Search",
                   func=faiss_search,
                   description="Useful for searching through the document store based on keywords."
                   )

lookup_tool = Tool(name="Lookup",
                   func=faiss_search,
                   description="Useful for looking up specific information or facts."
                   )

tools = [search_tool, lookup_tool]

llm = OpenAI(temperature=0.9)

react_1 = initialize_agent(tools,llm,agent=AgentType.REACT_DOCSTORE,verbose=True)

<ipython-input-7-acbca4e2210a>:18: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  llm = OpenAI(temperature=0.9)
<ipython-input-7-acbca4e2210a>:20: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use :meth:`~Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc.` instead.
  react_1 = initialize_agent(tools,llm,agent=AgentType.REACT_DOCSTORE,verbose=True)


In [8]:
question = "what is National Ataxia Foudnations's revenue 2023"
react_1.run(question)

<ipython-input-8-ba373a85542a>:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  react_1.run(question)




> Entering new AgentExecutor chain...
Thought: I need to search National Ataxia Foundation, find its revenue in 2023.
Action: Search[National Ataxia Foundation]
Observation: [Document(metadata={}, page_content='Note 1:  Summary of Significant Accounting Policies  \n \nA. Nature of Activities  \n \nNational Ataxia Foundation  (the Foundation) was incorporated as a Minnesota nonprofit corporation in 1957. The \nFoundation’s mission is to accelerate the development of treatments and a cure while working to improve the lives of \nthose living with Ataxia.'), Document(metadata={}, page_content='and who intend careers as scientists or other clinician -scientists within the field of ataxia.  \n \n6. National Ataxia Foundation Graduate Research Fellowship: The National Ataxia Foundation Graduate \nResearch Fellowship is a competitive, non -renewable, merit -based award intended to encourage pre -doctoral \nstudents to pursue research and a career in the field  of ataxia.')]
Thought: The reve

'contact National Ataxia Foundation directly for their revenue in 2023'

# 4. Using create_react_agent

In [24]:
from langchain import hub
from langchain_openai import OpenAI
from langchain_experimental.utilities import PythonREPL
from langchain_core.tools import Tool
from langchain.agents import create_react_agent
from langchain.agents import AgentExecutor


In [ ]:
# for calculation tool
# python_repl = PythonREPL()
    # PythonREPLTool(name="Calculator", description="Useful for performing complex calculations.")
    # Tool(
    #     name="calculate",
    #     description="Useful for performing complex calculations",
    #     func=python_repl.run,
    #     )

In [48]:
prompt = hub.pull("hwchase17/react")

tools = [
    Tool(
        name="Search",
        func=faiss_search,
        description="Useful for searching through the document store based on keywords."
    ),
    Tool(name="Lookup",
        func=faiss_search,
        description="Useful for looking up specific information or facts."
    ),
]

llm = OpenAI(
    temperature=0.9,

    )

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
def react2_run_agent(input_text):
    result = agent_executor.invoke({"input": input_text})
    print(result)

question = "what is National Ataxia Foudnations's revenue 2023"
react2_run_agent(question)


/usr/local/lib/python3.10/dist-packages/langsmith/client.py:323: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(




> Entering new AgentExecutor chain...
 Consider searching for National Ataxia Foudnation's financial reports from past years to find information about their projected revenue in 2023.
Action: Search
Action Input: National Ataxia Foudnation financial reports[Document(metadata={}, page_content="Annual Financial  \nReport  \nNational Ataxia Foundation   \nSt. Louis Park, Minnesota  \n \n \nFor the years ended December 31, 2023  and 2022  \n National Ataxia Foundation  \nTable of Contents  \nDecember 31, 2023  and 2022  \nPage No.  \nIndependent Auditor's Report  3 \nFinancial Statements  \nStatement s of Financial Position  6 \nStatement s of Activities  7 \nStatements of Functional Expenses  9"), Document(metadata={}, page_content='Board of Directors  \nNational Ataxia Foundation  \nSt. Louis Park , Minnesota  \n \nOpinion  \n \nWe have audited the accompanying financial statements of National Ataxia Foundation  (the Foundation ), which comprise \nthe statements of financial position a

# 5. Using langGraph

In [11]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

tools = [
    Tool(
        name="Search",
        func=faiss_search,
        description="Useful for searching through the document store based on keywords."
    ),
    Tool(name="Lookup",
        func=faiss_search,
        description="Useful for looking up specific information or facts."
    ),
]


In [12]:
system_prompt = "You are a financial expert in underwriting and reading auditor notes."
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.1,)
graph = create_react_agent(llm, tools, state_modifier=system_prompt)

def react3_run_agent(question):
    inputs = {"messages": [("user", question)]}
    results=graph.stream(inputs, stream_mode="values");
    for s in results:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [13]:
question = "what is National Ataxia Foudnations's revenue 2023"
react3_run_agent(question)

================================ Human Message =================================

what is National Ataxia Foudnations's revenue 2023
================================== Ai Message ==================================
Tool Calls:
  Lookup (call_aQ4PJAnVKBnNbamI6wBOJmNq)
 Call ID: call_aQ4PJAnVKBnNbamI6wBOJmNq
  Args:
    __arg1: National Ataxia Foundation revenue 2023
================================= Tool Message =================================
Name: Lookup

[Document(metadata={}, page_content="Total Expenses 1,837,520$     579,104 $         943,904 $         3,360,528$      377,042 $         393,619 $         4,131,189$      National Ataxia Foundation\nStatements of Functional Expenses (Continued)\nFor the Years Ended December 31, 2023 and 2022\n2022\nProgram Services Supporting Services\nSee Independent Auditor's Report and Notes to the Financial Statements.\n102023 2022"), Document(metadata={}, page_content="Ending Net Assets 934,379 $        2,680,567$      3,614,946$      2023Natio

# 6. Focusing on the second method to explore

## 6.1 compare ReAct with simple prompting
* this is difficult to achieve, the second method only works with `gpt-3.5-turbo-instruct`, because `gpt-3.5-turbo-instruct` has a limit of input tokens around 4k, so passing the entire pdf text into the prompt is impossible, which can only be achieved by using `gpt-4-turbo`, but that would not be a fair comparison. Nontheless, here is the code for it.



In [18]:
# for the second method
question = "what is National Ataxia Foudnations's total revenue in 2023"
react2_run_agent(question)



> Entering new AgentExecutor chain...
 I should search for "National Ataxia Foundation revenue 2023"
Action: Search
Action Input: "National Ataxia Foundation revenue 2023"[Document(metadata={}, page_content="Total Expenses 1,837,520$     579,104 $         943,904 $         3,360,528$      377,042 $         393,619 $         4,131,189$      National Ataxia Foundation\nStatements of Functional Expenses (Continued)\nFor the Years Ended December 31, 2023 and 2022\n2022\nProgram Services Supporting Services\nSee Independent Auditor's Report and Notes to the Financial Statements.\n102023 2022"), Document(metadata={}, page_content="Ending Net Assets 934,379 $        2,680,567$      3,614,946$      2023National Ataxia Foundation\nStatements of Activities\nFor the Years Ended December 31, 2023 and 2022\nSee Independent Auditor's Report and Notes to the Financial Statements.\n7Without Donor With Donor\nRestriction Restriction Total\nSupport and Revenue\nSupport")]I should look specifically for

In [19]:
# using a big prompt directly without ReAct and RAG
# parse PDF into text first
from PyPDF2 import PdfReader
pdf_path = '/content/naf23.pdf'
def extract_text_from_pdf(pdf_path):
    pdf_reader = PdfReader(pdf_path)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()
    return text
auditor_note_text = extract_text_from_pdf(pdf_path)

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [22]:
llm = ChatOpenAI(temperature=0.9, model_name="gpt-4-turbo")
prompt = PromptTemplate.from_template(
    "You are a financial expert. Given the following auditor's note: {context}. "
    "Answer this question: {question}."
)
chain = prompt | llm
question = "what is National Ataxia Foudnations's total revenue in 2023"

def prompt_run_agent(question):
    response = chain.invoke(
        {
            "context": auditor_note_text,
            "question":question
        }
    )
    print(response)
prompt_run_agent(question)


content="The National Ataxia Foundation's total revenue in 2023 was $4,184,787." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 10735, 'total_tokens': 10756, 'completion_tokens_details': {'reasoning_tokens': 0}, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_68a5bb159e', 'finish_reason': 'stop', 'logprobs': None} id='run-e54d5b4b-6517-40d9-a8c8-fda2bb33e4a3-0' usage_metadata={'input_tokens': 10735, 'output_tokens': 21, 'total_tokens': 10756}


**both are wrong**
* `$3,614,946` is total assets
* `$4,184,787` is support and revenue combined

## 6.2 test the second method, focusing on not numbers, because numbers are not parsed perfectly, also include a few examples passing the entire pdf directly

### react performs ok with very simple plain text quesitons, not so ok with a little bit reasoning

In [30]:
react2_run_agent("how are National Ataxia Foudnations annual grants selected")



> Entering new AgentExecutor chain...
 In order to find the answer, we should search through the document store with the keywords "National Ataxia Foundation" and "grants" to see if there are any specific selection processes mentioned.
Action: Search
Action Input: "National Ataxia Foundation grants selection process"[Document(metadata={}, page_content='clinicians and scientists into  ataxia research, and help drive the field towards developing treatments and a cure for \nataxia. Annual grants are selected through a competitive review process that includes NAF’s Medical and Research \nAdvisory Board members and over 80 leading ataxia scien tists and clinicians, both domestic and international.'), Document(metadata={}, page_content='and who intend careers as scientists or other clinician -scientists within the field of ataxia.  \n \n6. National Ataxia Foundation Graduate Research Fellowship: The National Ataxia Foundation Graduate \nResearch Fellowship is a competitive, non -renewable,

In [39]:
react2_run_agent("what is National Ataxia Foudnations")



> Entering new AgentExecutor chain...
 I'm not sure, but it sounds like it might be an organization or foundation
Action: Lookup
Action Input: National Ataxia Foundations[Document(metadata={}, page_content='Note 1:  Summary of Significant Accounting Policies  \n \nA. Nature of Activities  \n \nNational Ataxia Foundation  (the Foundation) was incorporated as a Minnesota nonprofit corporation in 1957. The \nFoundation’s mission is to accelerate the development of treatments and a cure while working to improve the lives of \nthose living with Ataxia.'), Document(metadata={}, page_content='and who intend careers as scientists or other clinician -scientists within the field of ataxia.  \n \n6. National Ataxia Foundation Graduate Research Fellowship: The National Ataxia Foundation Graduate \nResearch Fellowship is a competitive, non -renewable, merit -based award intended to encourage pre -doctoral \nstudents to pursue research and a career in the field  of ataxia.')] It looks like it is a

In [43]:
react2_run_agent("National Ataxia Foudnations purchases a new euqipment for $5k, is it considered to be property and equipment")
# didn't find the correct context through search



> Entering new AgentExecutor chain...
 Yes, we need to determine if the new equipment fits within the definition of property and equipment.
Action: Search
Action Input: "National Ataxia Foundations equipment property and equipment definition"[Document(metadata={}, page_content='National Ataxia Foundation  \nNotes to the Financial Statements  \nDecember 31, 2023  and 2022  \n \nNote 1:  Summary of Significant Accounting Policies (Continued)  \n \nJ. Property and Equipment  \n \nProperty and equipment are capitalized at cost. Depreciation is provided for using the straight -line method over the'), Document(metadata={}, page_content='Note 1:  Summary of Significant Accounting Policies  \n \nA. Nature of Activities  \n \nNational Ataxia Foundation  (the Foundation) was incorporated as a Minnesota nonprofit corporation in 1957. The \nFoundation’s mission is to accelerate the development of treatments and a cure while working to improve the lives of \nthose living with Ataxia.')] Based on 

In [44]:
react2_run_agent("how much is an item considered to be equipment in National Ataxia Foudnations accounting rules")
# didn't find the correct context through search



> Entering new AgentExecutor chain...
 First, I should think about what I need to know to answer this question.
Action: Search
Action Input: "National Ataxia Foundation accounting equipment rules"[Document(metadata={}, page_content='Note 1:  Summary of Significant Accounting Policies  \n \nA. Nature of Activities  \n \nNational Ataxia Foundation  (the Foundation) was incorporated as a Minnesota nonprofit corporation in 1957. The \nFoundation’s mission is to accelerate the development of treatments and a cure while working to improve the lives of \nthose living with Ataxia.'), Document(metadata={}, page_content='National Ataxia Foundation  \nNotes to the Financial Statements  \nDecember 31, 2023  and 2022  \n \nNote 1:  Summary of Significant Accounting Policies (Continued)  \n \nJ. Property and Equipment  \n \nProperty and equipment are capitalized at cost. Depreciation is provided for using the straight -line method over the')] Now that I have some information on the National Ataxia

In [46]:
prompt_run_agent("National Ataxia Foudnations purchases a new euqipment for $5k, is it considered to be property and equipment")
# directly passing the entire pdf into it can work!!!

content='Based on the summary of significant accounting policies provided in the National Ataxia Foundation\'s financial statements, particularly under "J. Property and Equipment," the Foundation capitalizes items that have a cost greater than $2,500 and a useful life greater than one year. Since the new equipment purchased for $5,000 fits these criteria, it would indeed be considered property and equipment and would be capitalized on the Foundation\'s balance sheet. It would then be subject to depreciation over its estimated useful life according to the straight-line method, as stated in their accounting policies.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 10745, 'total_tokens': 10857, 'completion_tokens_details': {'reasoning_tokens': 0}, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_82bed303cf', 'finish_reason': 'stop', 'logprobs': None} id='run-

### react performs badly with numbers related questions

In [38]:
react2_run_agent("how much does National Ataxia Foudnations pay for tax in 2023")
# the answer is wrong, that number is total expense



> Entering new AgentExecutor chain...
 I should first search for information related to National Ataxia Foundation and their tax payments.
Action: Search
Action Input: "National Ataxia Foundation tax payments 2023"[Document(metadata={}, page_content="Annual Financial  \nReport  \nNational Ataxia Foundation   \nSt. Louis Park, Minnesota  \n \n \nFor the years ended December 31, 2023  and 2022  \n National Ataxia Foundation  \nTable of Contents  \nDecember 31, 2023  and 2022  \nPage No.  \nIndependent Auditor's Report  3 \nFinancial Statements  \nStatement s of Financial Position  6 \nStatement s of Activities  7 \nStatements of Functional Expenses  9"), Document(metadata={}, page_content="Total Expenses 1,837,520$     579,104 $         943,904 $         3,360,528$      377,042 $         393,619 $         4,131,189$      National Ataxia Foundation\nStatements of Functional Expenses (Continued)\nFor the Years Ended December 31, 2023 and 2022\n2022\nProgram Services Supporting Services\n

In [47]:
prompt_run_agent("how much does National Ataxia Foudnations pay for tax in 2023")
# not entirely correct, but much better than the react agent, NAF still pays payroll taxes

content='The National Ataxia Foundation is exempt from income taxes under Section 501(c)(3) of the Internal Revenue Code and is also exempt from Federal and State income taxes. Therefore, the National Ataxia Foundation did not pay any income taxes in 2023.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 10736, 'total_tokens': 10790, 'completion_tokens_details': {'reasoning_tokens': 0}, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'gpt-4-turbo-2024-04-09', 'system_fingerprint': 'fp_68a5bb159e', 'finish_reason': 'stop', 'logprobs': None} id='run-14b4d87b-aafc-4b57-b81d-ab2bf1641453-0' usage_metadata={'input_tokens': 10736, 'output_tokens': 54, 'total_tokens': 10790}


In [40]:
react2_run_agent("National Ataxia Foudnations's total expense increased from 4,131,189 in 2022 to 4,184,787 in 2023, do you know why, what caused it")
# this is only partially the reason, a lot of other expenses also increased



> Entering new AgentExecutor chain...
 I should try to find information on National Ataxia Foundation's financial reports or statements
Action: Lookup
Action Input: National Ataxia Foundation financial report or statement[Document(metadata={}, page_content="Annual Financial  \nReport  \nNational Ataxia Foundation   \nSt. Louis Park, Minnesota  \n \n \nFor the years ended December 31, 2023  and 2022  \n National Ataxia Foundation  \nTable of Contents  \nDecember 31, 2023  and 2022  \nPage No.  \nIndependent Auditor's Report  3 \nFinancial Statements  \nStatement s of Financial Position  6 \nStatement s of Activities  7 \nStatements of Functional Expenses  9"), Document(metadata={}, page_content='Board of Directors  \nNational Ataxia Foundation  \nSt. Louis Park , Minnesota  \n \nOpinion  \n \nWe have audited the accompanying financial statements of National Ataxia Foundation  (the Foundation ), which comprise \nthe statements of financial position as of December 31, 2023  and 2022 , a